In [1]:
import os
from pathlib import Path

_raiz = Path.cwd().resolve()
while not (_raiz / "data" / "gold").is_dir() and _raiz != _raiz.parent:
    _raiz = _raiz.parent
os.chdir(_raiz / "modelos")
print("cwd:", Path.cwd())

GOLD = _raiz / "data" / "gold"
TEMP = _raiz / "data_temp"

cwd: C:\Users\Powan\Desktop\Maestria\TFM\edev_models\modelos


In [2]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170)

TRAIN_END = pd.Timestamp("2024-12-31")
VAL_END = pd.Timestamp("2025-12-31")
TARGET_COLS = [f"price_h{h:02d}" for h in range(24)]

LISTON_MAE = 19.94        # persistencia_d1 (F11_baselines.ipynb)
LISTON_CAPTURA = 91.2     # media_movil_7d (F11_baselines.ipynb)

SEMILLA_BASE = 42          # misma convención que entrenar_finales.py (SEMILLA + s)
SEMILLAS = 3

# Referencias de corridas anteriores, para comparar en los mismos gráficos.
REF_HORARIO = {
    "XGBoost (horario)": {"MAE_val": 12.99, "captura_val": 94.96},
    "LightGBM (horario)": {"MAE_val": 13.25, "captura_val": 94.54},
}
REF_REDES_MAE = {
    "denso (redes)": 12.62,
    "boosting=LightGBM (redes)": 13.28,
    "gru (redes)": 13.38,
    "seq2seq_absoluto (redes)": 14.89,
    "ENSEMBLE (redes, 8 familias)": 11.86,
}

print("Constantes cargadas.")

Constantes cargadas.


In [4]:
ruta_csv = GOLD / "matriz_nucleo.csv"
df = pd.read_csv(ruta_csv, parse_dates=["fecha_pred", "fecha_objetivo", "ts"])
print(f"{df.shape[0]} filas x {df.shape[1]} columnas -- nulos: {df.isna().sum().sum()}")

CONTROL = ["fecha_pred", "fecha_objetivo", "ts", "split", "hora"]
TARGET = "target_price"
feature_cols = [c for c in df.columns if c not in CONTROL + [TARGET]]
print(f"{len(feature_cols)} columnas de features")
print(df["split"].value_counts())

mask_train = df["split"] == "train"
mask_resto = df["split"].isin(["validation", "test"])

X_train = df.loc[mask_train, feature_cols]
y_train = df.loc[mask_train, TARGET]
X_resto = df.loc[mask_resto, feature_cols]

# Precio real en formato ancho (fecha_objetivo x hora) -- para reusar errores()/ingreso_arbitraje().
P = df.pivot_table(index="fecha_objetivo", columns="hora", values="target_price")
P.columns = [f"price_h{int(h):02d}" for h in P.columns]
P = P[TARGET_COLS]
print(f"P (precio ancho): {P.shape}")


def pivotear_pred(pred_serie: pd.Series, df_resto: pd.DataFrame) -> pd.DataFrame:
    tmp = df_resto[["fecha_objetivo", "hora"]].copy()
    tmp["pred"] = pred_serie.values
    wide = tmp.pivot_table(index="fecha_objetivo", columns="hora", values="pred")
    wide.columns = [f"price_h{int(h):02d}" for h in wide.columns]
    return wide.reindex(columns=TARGET_COLS)

57521 filas x 133 columnas -- nulos: 0
127 columnas de features
split
train         43699
validation     8759
test           5063
Name: count, dtype: int64
P (precio ancho): (2397, 24)


In [7]:
import pandas as pd

# Create a complete hourly range based on your min and max dates
full_range = pd.date_range(start=df['ts'].min(), 
                           end=df['ts'].max(), 
                           freq='h')

# Find the difference
missing_dates = full_range.difference(df['ts'])

print(f"Total missing hours: {len(missing_dates)}")
if len(missing_dates) > 0:
    print(missing_dates)

Total missing hours: 7
DatetimeIndex(['2020-03-29 02:00:00', '2021-03-28 02:00:00', '2022-03-27 02:00:00', '2023-03-26 02:00:00', '2024-03-31 02:00:00', '2025-03-30 02:00:00',
               '2026-03-29 02:00:00'],
              dtype='datetime64[us]', freq=None)
